# Attention Block
## Causal Attention

In [16]:
import torch
import torch.nn as nn

class SelfAttentionV3(nn.Module):
    def __init__(self, d_in, d_out, context_length, kqv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=kqv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=kqv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=kqv_bias)
        self.context_length = context_length
    
    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5,
            dim = -1
        )
        context_len = attn_scores.shape[0]
        mask = torch.tril(torch.ones(context_len, context_len))
        
        masked_attn_weights = mask * attn_weights
        row_sums = masked_attn_weights.sum(dim=-1, keepdim=True)
        masked_norm_attn_weights = masked_attn_weights / row_sums
        
        context_vec = masked_norm_attn_weights @ values
        
        return context_vec


In [ ]:
class SelfAttentionV3_2(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout=0.5, kqv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=kqv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=kqv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=kqv_bias)
        self.dropout = nn.Dropout(dropout)
        self.context_length = context_length
    
    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        causal_mask = torch.triu(torch.ones(self.context_length, self.context_length), diagonal=1)
        # print(f"causal_mask => {causal_mask}")
        masked_attn_scores = torch.masked_fill(attn_scores, causal_mask.bool(), -torch.inf)
        masked_attn_weights = torch.softmax(
            masked_attn_scores / keys.shape[-1] ** 0.5,
            dim = -1
        )
        
        context_vec = masked_attn_weights @ values
        
        return context_vec
        

In [40]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.context_length = context_length
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
            persistent=True
        )
    
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.transpose(-2, -1)
        masked_attn_scores = torch.masked_fill(attn_scores, self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        masked_attn_weights = torch.softmax(
            masked_attn_scores / keys.shape[-1] ** 0.5,
            dim=-1
        )
        
        context_vec = masked_attn_weights @ values
        
        return context_vec

In [43]:
import torch
import torch.nn as nn

d_in = 3
d_out = 2
context_length = 6
input = torch.rand(context_length, d_in)

print(f"input => {input} \n shape => {input.shape}")

torch.manual_seed(123)
sa_v3 = SelfAttentionV3(d_in, d_out, context_length)

print(sa_v3(input))


torch.manual_seed(123)
sa_v3_2 = SelfAttentionV3_2(d_in, d_out, context_length, dropout=0.5)

print(sa_v3_2(input))


batch = torch.stack((input, input), dim=0)
torch.manual_seed(123)
ca = CausalAttention(d_in, d_out, context_length, dropout=0.5)

print(ca(batch))

input => tensor([[0.3821, 0.6605, 0.8536],
        [0.5932, 0.6367, 0.9826],
        [0.2745, 0.6584, 0.2775],
        [0.8573, 0.8993, 0.0390],
        [0.9268, 0.7388, 0.7179],
        [0.7058, 0.9156, 0.4340]]) 
 shape => torch.Size([6, 3])
tensor([[-0.5995, -0.0116],
        [-0.6623,  0.0065],
        [-0.5824, -0.0675],
        [-0.6233, -0.1719],
        [-0.6783, -0.1652],
        [-0.6885, -0.1936]], grad_fn=<MmBackward0>)
tensor([[-0.5995, -0.0116],
        [-0.6623,  0.0065],
        [-0.5824, -0.0675],
        [-0.6233, -0.1719],
        [-0.6783, -0.1652],
        [-0.6885, -0.1936]], grad_fn=<MmBackward0>)
tensor([[[-0.5995, -0.0116],
         [-0.6623,  0.0065],
         [-0.5824, -0.0675],
         [-0.6233, -0.1719],
         [-0.6783, -0.1652],
         [-0.6885, -0.1936]],

        [[-0.5995, -0.0116],
         [-0.6623,  0.0065],
         [-0.5824, -0.0675],
         [-0.6233, -0.1719],
         [-0.6783, -0.1652],
         [-0.6885, -0.1936]]], grad_fn=<UnsafeViewB

In [23]:
import torch
import torch.nn as nn

bs = 2
context_length = 6
d_in = 3
d_out = 6
n_heads = 2
head_dim = d_out // n_heads

input = torch.rand([bs, context_length, d_in])
Wq = torch.rand([d_in, d_out])
Wk = torch.rand([d_in, d_out])
Wv = torch.rand([d_in, d_out])

q = input @ Wq
q = q.view(bs, context_length, n_heads, head_dim)
q = q.transpose(1,2)
print("Shape of q => ",q.shape) # 2,6,2

k = input @ Wk
k = k.view(bs, context_length, n_heads, head_dim)
k = k.transpose(1,2)
print("Shape of k => ",k.shape) # 2,6,2

v = input @ Wv
v = v.view(bs, context_length, n_heads, head_dim)
v = v.transpose(1,2)
print("Shape of v => ",v.shape) # 2,6,2

attn_score = q @ k.transpose(2,3)
print("Shape of attn score => ", attn_score.shape)

causal_mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked_attn_scores = attn_score.masked_fill(causal_mask.bool(), -torch.inf)

attn_weights = torch.softmax(
    masked_attn_scores/k.shape[-1]**0.5, 
    dim=-1
)

context_vec = attn_weights @ v
context_vec.transpose_(1,2)
print(f"Shape fo context vec => {context_vec.shape}")

Shape of q =>  torch.Size([2, 2, 6, 3])
Shape of k =>  torch.Size([2, 2, 6, 3])
Shape of v =>  torch.Size([2, 2, 6, 3])
Shape of attn score =>  torch.Size([2, 2, 6, 6])
Shape fo context vec => torch.Size([2, 6, 2, 3])
